# GOIT pipelines summary sheets — June 2026 release

Produces summary tables (km by region, country, owner, start year) for the GOIT pipelines
data release. Writes a single Excel file that is pasted into the shared summary tables:
https://docs.google.com/spreadsheets/d/1OYH6D7c-D0FsL5GzBGijtkmvQCTkBUclj-UVoOieUFo/edit


## imports and configuration

In [77]:
%pip install -q -e ../../../gem-tracker-constants

import datetime
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pygsheets

# Canonical fuel buckets, status orderings, and gas-and-hydrogen collapse helper.
# Source of truth: the in-repo gem-tracker-constants package (repo root,
# installed editable above).
from gem_tracker_constants import (
    GAS_FUEL_OPTIONS,
    OIL_FUEL_OPTIONS,
    NGL_FUEL_OPTIONS,
    PIPELINE_STATUS as STATUS_LIST,
    PIPELINE_EXCEL_STATUS as EXCEL_STATUS_LIST,
    PIPELINE_IN_DEV_COL as IN_DEV_COL,
    collapse_gas_and_hydrogen,
)


Note: you may need to restart the kernel to use updated packages.


In [96]:
# === config ===========================================================
# Set FUEL_TYPE to one of: "Gas", "Oil", "NGL". Run the notebook once per fuel.
FUEL_TYPE = "Oil"
#FUEL_TYPE = "NGL"

# Source spreadsheet (June 2026 GOIT oil/NGL release).
# Past keys, for reference:
#   1WaBMIdfRWqSqXUw7_cKXo3RipyhPdnNN8flqEYfMZIA  — Dec 2023 gas
#   1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek  — CURRENT (rolling)
#   1OXybaZOn0f2ONB6d_J0A3SG2bJ660C2Kr8fuc5o8cjs  — Dec 2024 GGIT
#   1xjaeq0OwdN-Orht7Q7ynPHB2uY_gnMvAeha-QHkzoZw  — Nov 2025 GGIT
SPREADSHEET_KEY = "1gChRPYLrcirx3lNI_DHWs5GaKTtfXgELVg1bHix8zqs" # June 2026 GOIT

# Where to write the Excel summary. Defaults to alongside this notebook.
OUTPUT_DIR = Path.cwd()

# Region filter:
#   "Global"             — every country
#   "AsiaGasTracker"     — countries flagged AsiaGasTracker == "Yes"
#   "EuroGasTracker"     — countries flagged EuroGasTracker == "Yes"
#   "AfricaGasTracker"   — countries flagged AfricaGasTracker == "Yes"
#   "LatinAmericaTracker" — countries flagged LatinAmericaTracker == "Yes"
REGION_NAME = "Global"


In [97]:
# Fuel buckets are imported above from gem-tracker-constants.
# Per-run glue: pick the sheet to read, the bucket to filter on, and the output label.
FUEL_CONFIG = {
    "Gas": {"sheet": "Gas pipelines",     "options": GAS_FUEL_OPTIONS, "label": "Gas"},
    "Oil": {"sheet": "Oil/NGL pipelines", "options": OIL_FUEL_OPTIONS, "label": "Oil"},
    "NGL": {"sheet": "Oil/NGL pipelines", "options": NGL_FUEL_OPTIONS, "label": "NGL"},
}
assert FUEL_TYPE in FUEL_CONFIG, f"unknown FUEL_TYPE {FUEL_TYPE!r}"
FUEL_OPTIONS = FUEL_CONFIG[FUEL_TYPE]["options"]
FUEL_LABEL = FUEL_CONFIG[FUEL_TYPE]["label"]


In [98]:
# Status orderings (STATUS_LIST, EXCEL_STATUS_LIST, IN_DEV_COL) are imported above
# from gem-tracker-constants.

# Columns that should be numeric. pygsheets returns everything as strings, so we coerce
# at load time (a single source of truth) — this prevents the entire class of bug where
# an empty string slips past .notna() and crashes .quantile() / sort().
NUMERIC_COLS_PIPES = [
    "LengthMergedKm",
    "StartYearEarliest",
    "ProposalYear",
    "ConstructionYear",
    "ShelvedYear",
    "CancelledYear",
    "CostUSDPerKm",
    "CapacityBbl",
    "DiameterInches",
]
NUMERIC_COLS_RATIOS = [
    "LengthMergedKmByCountry",
    "LengthEstimateKmByCountry",  # may contain comma-formatted strings like "1,236.43"
    "LengthKnownKmByCountry",     # used by the cost-per-km estimate
    "LengthPerCountryFraction",
    "CostUSDPerKm",
]


## load the source workbook

In [99]:
gc = pygsheets.authorize(service_account_env_var="GDRIVE_API_CREDENTIALS")
spreadsheet = gc.open_by_key(SPREADSHEET_KEY)


def load_sheet(title: str, start: str | None = None) -> pd.DataFrame:
    """Read a worksheet into a DataFrame. `start` is the top-left A1 cell when the
    sheet has header rows above the data (e.g. "A3" skips two prefix rows)."""
    ws = spreadsheet.worksheet("title", title)
    return ws.get_as_df(start=start) if start else ws.get_as_df()


def coerce_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Convert listed columns to numeric. Strips thousands separators (commas) first,
    then `pd.to_numeric(errors="coerce")` turns anything unparseable into NaN.
    Columns absent from `df` are silently skipped."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        s = df[col]
        if s.dtype == object:
            s = s.astype(str).str.replace(",", "", regex=False)
        df[col] = pd.to_numeric(s, errors="coerce")
    return df


pipes_sheet_title = FUEL_CONFIG[FUEL_TYPE]["sheet"]
pipes_df_orig = load_sheet(pipes_sheet_title, start="A3").drop(
    columns="WKTFormat", errors="ignore"
)
country_ratios_df = load_sheet("Country ratios by pipeline").drop(
    columns="WKTFormat", errors="ignore"
)
region_df_orig = load_sheet("Country dictionary", start="A2")

# coerce numeric columns at load time — single source of truth for type cleanup
pipes_df_orig = coerce_numeric(pipes_df_orig, NUMERIC_COLS_PIPES)
country_ratios_df = coerce_numeric(country_ratios_df, NUMERIC_COLS_RATIOS)


## clean inputs

In [100]:
def clean_placeholders(df: pd.DataFrame) -> pd.DataFrame:
    """Replace `--` and empty-string sentinels with NaN."""
    return df.replace({"--": np.nan, "": np.nan})


# Same row filters as the release-downloads export
# (releases/downloads/convert-ggit-goit-to-tracker-release-downloads.ipynb,
# load_pipeline_data), so these summaries calculate on exactly the rows shipped
# in the data files.
pipes_df_orig = pipes_df_orig.loc[
    (pipes_df_orig["Status"] != "N/A")
    & (pipes_df_orig["PipelineName"] != "")
    & (pipes_df_orig["RouteAccuracy"] != "")
].copy()

# subset pipes to the chosen fuel bucket; collapse "Gas and Hydrogen" -> "Gas" for gas runs
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig["Fuel"].isin(FUEL_OPTIONS)].copy()
if FUEL_TYPE == "Gas":
    collapse_gas_and_hydrogen(pipes_df_orig)
    collapse_gas_and_hydrogen(country_ratios_df)

# the ratios tab needs the same status cleanup: rows with statuses outside
# STATUS_LIST ("N/A", "mixed status") fall out of the status pivots anyway,
# but the cost-per-km averages don't pivot by status and would otherwise
# include them
country_ratios_df = country_ratios_df.loc[
    country_ratios_df["Status"].isin(STATUS_LIST)
].copy()

# no Wiki filter on the ratios tab: pipelines without wiki pages still ship in
# the release downloads, so they count in the summaries too
country_ratios_df = clean_placeholders(country_ratios_df)

pipes_df_orig = clean_placeholders(pipes_df_orig)

## region selection and country / region subsets

Selects countries for the chosen `REGION_NAME` and subsets `country_ratios_df` and `pipes_df_orig`
to those countries. Combined in one cell so they can't be run out of order.

In [101]:
if REGION_NAME == "Global":
    region_df_touse = region_df_orig
else:
    if REGION_NAME not in region_df_orig.columns:
        raise KeyError(
            f"REGION_NAME={REGION_NAME!r} but region_df_orig has no column {REGION_NAME!r}; "
            f"available: {[c for c in region_df_orig.columns if c.endswith('Tracker')]}"
        )
    region_df_touse = region_df_orig.loc[region_df_orig[REGION_NAME] == "Yes"]

region_df_touse_cleaned = region_df_touse.loc[
    (region_df_touse["Region"] != "--") & (region_df_touse["SubRegion"] != "--")
]
multiindex_region_subregion = (
    region_df_touse_cleaned.groupby(["Region", "SubRegion"])["Country"].count().index
)

# isin() against an explicit set — no regex, no substring false-positives
countries_in_region = set(region_df_touse["Country"])

country_ratios_df_touse = country_ratios_df.loc[
    country_ratios_df["Country"].isin(countries_in_region)
].copy()
pipes_df_touse = pipes_df_orig.loc[
    pipes_df_orig["CountriesOrAreas"]
    .str.split(", ")
    .map(lambda cs: any(c in countries_in_region for c in cs))
].copy()

print(f"region={REGION_NAME!r}: {len(countries_in_region)} countries, "
      f"{len(pipes_df_touse)} pipeline rows, {len(country_ratios_df_touse)} ratio rows")
multiindex_region_subregion


region='Global': 272 countries, 1634 pipeline rows, 6847 ratio rows


MultiIndex([(  'Africa',                 'Northern Africa'),
            (  'Africa',              'Sub-Saharan Africa'),
            ('Americas', 'Latin America and the Caribbean'),
            ('Americas',                'Northern America'),
            (    'Asia',                    'Central Asia'),
            (    'Asia',                    'Eastern Asia'),
            (    'Asia',              'South-eastern Asia'),
            (    'Asia',                   'Southern Asia'),
            (    'Asia',                    'Western Asia'),
            (  'Europe',                  'Eastern Europe'),
            (  'Europe',                 'Northern Europe'),
            (  'Europe',                 'Southern Europe'),
            (  'Europe',                  'Western Europe'),
            ( 'Oceania',       'Australia and New Zealand'),
            ( 'Oceania',                       'Melanesia'),
            ( 'Oceania',                      'Micronesia'),
            ( 'Oceania',

## set up Excel writer

In [102]:
today = datetime.date.today().isoformat()
output_path = OUTPUT_DIR / f"GOIT-Summary-Sheets-{FUEL_LABEL}-{today}.xlsx"
print(f"writing to {output_path}")
excel_writer = pd.ExcelWriter(output_path)


writing to /Users/baird/Dropbox/_git_ALL/_github-repos-gem/goit-ggit-data-ops/scripts/data-release-summary-sheets/2026-q2-oil-pipelines/GOIT-Summary-Sheets-Oil-2026-06-15.xlsx


## km by region and km by country

In [103]:
country_ratios_df_subset = country_ratios_df_touse.loc[
    country_ratios_df_touse["Fuel"].isin(FUEL_OPTIONS)
]

country_list = sorted(country_ratios_df_subset["Country"].dropna().unique())


def pivot_km(group_cols, index) -> pd.DataFrame:
    """Sum LengthMergedKmByCountry by Status × group_cols, reshape to wide form."""
    grouped = (
        country_ratios_df_subset.groupby([*group_cols, "Status"])["LengthMergedKmByCountry"]
        .sum()
        .unstack("Status")
        .reindex(columns=STATUS_LIST)
        .reindex(index=index)
        .fillna(0)
    )
    grouped[IN_DEV_COL] = grouped[["proposed", "construction"]].sum(axis=1)
    return grouped[EXCEL_STATUS_LIST]


km_by_country = pivot_km(["Country"], country_list)
km_by_country.index.name = "Country"

km_by_region = pivot_km(["Region", "SubRegion"], multiindex_region_subregion)
km_by_region.index.names = ["Region", "Subregion"]

km_by_country.loc["Total"] = km_by_country.sum(axis=0).values
# full 2-level key — a flat "Total" label would collapse the MultiIndex to tuples
km_by_region.loc[("Total", ""), :] = km_by_region.sum(axis=0).values

# drop countries with no km in any status, then blank out zeros for display
km_by_country = km_by_country.loc[~(km_by_country == 0).all(axis=1)]
km_by_country = km_by_country.replace(0, "")
km_by_region = km_by_region.replace(0, "")

km_by_region.to_excel(excel_writer, sheet_name="Kilometers by region")
km_by_country.to_excel(excel_writer, sheet_name="Kilometers by country")
km_by_region


Status                                    proposed construction  \
Region   Subregion                                                
Africa   Northern Africa                     297.0       1403.0   
         Sub-Saharan Africa                4588.26       1443.0   
Americas Latin America and the Caribbean    644.01                
         Northern America                  2066.54       636.49   
Asia     Central Asia                      2278.78                
         Eastern Asia                      4465.13      1691.62   
         South-eastern Asia                                       
         Southern Asia                       103.0      4126.54   
         Western Asia                      4468.88       965.46   
Europe   Eastern Europe                     968.52      2006.74   
         Northern Europe                                          
         Southern Europe                    232.22                
         Western Europe                                           
Oceania  Australia and New Zealand                                
         Melanesia                                                
         Micronesia                                               
         Polynesia                                                
Total                                     20112.34     12272.85   

Status                                   in development (proposed + construction)  \
Region   Subregion                                                                  
Africa   Northern Africa                                                   1700.0   
         Sub-Saharan Africa                                               6031.26   
Americas Latin America and the Caribbean                                   644.01   
         Northern America                                                 2703.03   
Asia     Central Asia                                                     2278.78   
         Eastern Asia                                                     6156.75   
         South-eastern Asia                                                         
         Southern Asia                                                    4229.54   
         Western Asia                                                     5434.34   
Europe   Eastern Europe                                                   2975.26   
         Northern Europe                                                            
         Southern Europe                                                   232.22   
         Western Europe                                                             
Oceania  Australia and New Zealand                                                  
         Melanesia                                                                  
         Micronesia                                                                 
         Polynesia                                                                  
Total                                                                    32385.19   

Status                                     shelved cancelled  operating  \
Region   Subregion                                                        
Africa   Northern Africa                    556.48     620.0   20931.07   
         Sub-Saharan Africa                 498.11    1500.0   13513.16   
Americas Latin America and the Caribbean     130.0    2802.0   27371.89   
         Northern America                           40580.49  116819.12   
Asia     Central Asia                                 801.26    9204.43   
         Eastern Asia                      3934.64   2867.57    35476.7   
         South-eastern Asia                            306.0    2321.57   
         Southern Asia                       220.0   7247.45   20398.51   
         Western Asia                      3408.56   1672.64   23473.67   
Europe   Eastern Europe                     436.87   4889.29   60315.55   
         Northern Europe                                  

In [104]:
# === consistency guard: pipes tab vs country-ratios tab =====================
# The km tables sum LengthMergedKmByCountry from the ratios tab; the quick
# stats and the release downloads sum LengthMergedKm from the pipes tab. The
# two drift when the backend hasn't refreshed one of them — fail loudly
# instead of shipping a silent gap.
GUARD_TOL_KM = 25

guard = pd.DataFrame({
    "pipes_km": pipes_df_touse.groupby("Status")["LengthMergedKm"].sum(),
    "ratios_km": country_ratios_df_subset.groupby("Status")["LengthMergedKmByCountry"].sum(),
}).reindex(STATUS_LIST).fillna(0)
guard["diff_km"] = guard["pipes_km"] - guard["ratios_km"]
print(guard.round(1).to_string())

bad = guard.loc[guard["diff_km"].abs() > GUARD_TOL_KM]
if not bad.empty:
    # name the offenders so the failure is actionable; grouping by
    # (Wiki, PipelineName) keeps empty-Wiki pipelines from collapsing
    # into one anonymous group
    keys = ["Wiki", "PipelineName", "Status"]
    offenders = (
        pipes_df_touse.groupby(keys, dropna=False)["LengthMergedKm"].sum()
        .subtract(
            country_ratios_df_subset.groupby(keys, dropna=False)["LengthMergedKmByCountry"].sum(),
            fill_value=0,
        )
    )
    offenders = offenders.loc[offenders.abs() > 1].sort_values(key=abs, ascending=False)
    print("\nper-pipeline mismatches (>1 km), pipes minus ratios:")
    for (wiki, name, status), diff in offenders.head(20).items():
        print(f"  {diff:>+9,.1f} km  {status:<13} {name}")
    if REGION_NAME == "Global":
        raise AssertionError(
            f"pipes and ratios tabs disagree by >{GUARD_TOL_KM} km for {list(bad.index)} "
            "— backend ratios refresh needed before release"
        )
    print(f"\nREGION_NAME={REGION_NAME!r}: totals can differ legitimately — pipes keeps "
          "whole multi-country pipelines, ratios keeps only in-region country rows")

              pipes_km  ratios_km  diff_km
Status                                    
proposed       20112.4    20112.3      0.0
construction   12272.8    12272.8      0.0
shelved        10321.8    10321.8      0.0
cancelled      65259.5    65259.5      0.0
operating     346475.8   346475.8      0.0
idle            5175.0     5175.0      0.0
mothballed      5874.9     5874.9      0.0
retired        21325.0    21325.0      0.0


## km by parent company

Each pipeline row has a `Parent` string like `"TC Energy Corp [60%]; Sempra Energy [40%]"`.
We parse this into per-owner rows, weight the per-country km by the ownership fraction, then
pivot to status columns.

In [105]:
CJK_RE = re.compile(r"[\u4e00-\u9fff]+")
PERCENT_RE = re.compile(r"\d+(?:\.\d+)?%")
BRACKETS_RE = re.compile(r" \[.*?\]")


def parse_parent_string(parent_string) -> tuple[list[str], list[float]]:
    """Split a Parent cell into ([owner, ...], [fraction, ...]). Empty/NaN parent
    returns (["unknown"], [1.0]). If a percentage is missing, the leftover fraction is
    divided evenly across the owners without one."""
    if parent_string is None or (isinstance(parent_string, float) and np.isnan(parent_string)):
        return ["unknown"], [1.0]
    parent_string = str(parent_string).strip()
    if not parent_string:
        return ["unknown"], [1.0]

    # non-researched QCC owner: leading CJK character with a [100.00%] suffix; keep verbatim
    if CJK_RE.match(parent_string[:1]) and parent_string.endswith("[100.00%]"):
        return [parent_string.removesuffix(" [100.00%]")], [1.0]

    parents = BRACKETS_RE.sub("", parent_string).split("; ")
    pcts = [float(m.rstrip("%")) / 100.0 for m in PERCENT_RE.findall(parent_string)]

    if len(parents) != len(pcts):
        if not pcts:
            pcts = [1.0 / len(parents)] * len(parents)
        else:
            n_missing = len(parents) - len(pcts)
            leftover = 1.0 - float(np.nansum(pcts))
            pcts = pcts + [leftover / n_missing] * n_missing
    return parents, pcts


In [106]:
# guard: every row in our subset must have a ProjectID
missing_pid = country_ratios_df_subset["ProjectID"].isna()
if missing_pid.any():
    raise ValueError(
        f"missing ProjectID in rows: {country_ratios_df_subset.index[missing_pid].tolist()}"
    )

# build owner-parent rows as a list of dicts, then one DataFrame at the end
# (avoids O(n²) repeated pd.concat in a loop)
records = []
for row in country_ratios_df_subset.itertuples(index=False):
    parents, pcts = parse_parent_string(row.Parent)
    for parent, frac in zip(parents, pcts):
        records.append(
            {
                "Parent": parent,
                "ProjectID": row.ProjectID,
                "FractionOwnership": frac,
                "Country": row.Country,
                "Status": row.Status,
                "LengthMergedKmByCountry": row.LengthMergedKmByCountry,
            }
        )

owner_parent_calculations_df = pd.DataFrame.from_records(records)
owner_parent_calculations_df["KmOwnership"] = (
    owner_parent_calculations_df["FractionOwnership"]
    * owner_parent_calculations_df["LengthMergedKmByCountry"]
)
owner_parent_calculations_df


,Parent,ProjectID,FractionOwnership,Country,Status,LengthMergedKmByCountry,KmOwnership
0,Enbridge Inc,P0001,1.0000,Canada,operating,1209.30,1209.300000
1,Enbridge Inc,P0001,1.0000,United States,operating,580.70,580.700000
2,Enbridge Inc,P0002,0.8843,Canada,operating,545.57,482.447551
3,unknown,P0002,0.1157,Canada,operating,545.57,63.122449
4,Enbridge Inc,P0004,1.0000,Canada,operating,155.19,155.190000
...,...,...,...,...,...,...,...
2499,unknown,P7987,1.0000,United States,construction,225.31,225.310000
2500,unknown,P7988,1.0000,United States,construction,160.93,160.930000
2501,unknown,P7989,1.0000,United States,construction,96.56,96.560000
2502,unknown,P7996,1.0000,Iraq,construction,70.00,70.000000


In [107]:
owners_km_by_status_df = (
    owner_parent_calculations_df.groupby(["Parent", "Status"])["KmOwnership"]
    .sum()
    .unstack("Status")
    .reindex(columns=STATUS_LIST)
)
owners_km_by_status_df[IN_DEV_COL] = owners_km_by_status_df[["proposed", "construction"]].sum(axis=1)
owners_km_by_status_df = owners_km_by_status_df[EXCEL_STATUS_LIST]

owners_km_by_status_df.loc["Total"] = owners_km_by_status_df.sum(axis=0, min_count=0).values

owners_km_by_status_df = owners_km_by_status_df.replace({np.nan: "", 0: ""})
owners_km_by_status_df.to_excel(excel_writer, sheet_name="Kilometers by owner")
owners_km_by_status_df


Status,proposed,construction,in development (proposed + construction),shelved,cancelled,operating,idle,mothballed,retired
Parent,,,,,,,,,
1832 Asset Management LP,,,,,,78.272,,,
APA Corp,,,,,,176.838,,,
API Holding SpA,,,,,,146.0,,,
ARB Midstream LLC,,,,,,149.47,,,
Abu Dhabi National Energy Company PJSC,,,,,,153.0,,,
...,...,...,...,...,...,...,...,...,...
natural person(s),,,,,16.56,16.56,,,
small shareholder(s),,71.94,71.94,2.73,79.692,92.284476,,,
unknown,9379.72648,2298.92,11678.64648,8209.06,13807.693333,60701.889293,1147.35,150.0,9121.668


## km by start year, status

In [108]:
def km_by_year(status_values, year_col: str) -> pd.Series:
    """Sum LengthMergedKm for pipes with the given status(es), grouped by `year_col`.
    `pipes_df_touse` is already subset to the chosen fuel bucket at load time."""
    if isinstance(status_values, str):
        status_values = [status_values]
    subset = pipes_df_touse.loc[
        pipes_df_touse["Status"].isin(status_values)
        & pipes_df_touse[year_col].notna()  # drop missing-year rows so they don't form a NaN group
    ]
    # year cols were coerced to float at load time; cast index back to int for clean display
    grouped = subset.groupby(year_col)["LengthMergedKm"].sum()
    grouped.index = grouped.index.astype(int)
    return grouped


pipes_started_sum      = km_by_year("operating",    "StartYearEarliest")
pipes_construction_sum = km_by_year("construction", "ConstructionYear")
pipes_proposed_sum     = km_by_year("proposed",     "ProposalYear")

# derive the year range from the data, with a floor at 1980 and ceiling at 2025
all_years = pd.concat([pipes_started_sum, pipes_construction_sum, pipes_proposed_sum]).index
year_min = int(min(1980, all_years.min())) if len(all_years) else 1980
year_max = int(max(2025, all_years.max())) if len(all_years) else 2025
year_index = pd.Index(range(year_min, year_max + 1), name="Start year")

km_by_start_year = pd.DataFrame(index=year_index)
km_by_start_year[f"{FUEL_LABEL} pipeline km operating"]    = pipes_started_sum
km_by_start_year[f"{FUEL_LABEL} pipeline km construction"] = pipes_construction_sum
km_by_start_year[f"{FUEL_LABEL} pipeline km proposed"]     = pipes_proposed_sum
km_by_start_year = km_by_start_year.fillna(0)
km_by_start_year.loc["Total"] = km_by_start_year.sum(axis=0)

km_by_start_year.to_excel(excel_writer, sheet_name="Kilometers by start year")


## cost estimates

Builds two artifacts. First, regional and subregional mean `CostUSDPerKm`, drawn from the
**global** set of pipelines (by country-fraction) that have both a known length and a known
per-km cost, trimmed to the [2.5%, 97.5%] interquantile range of `CostUSDPerKm` to drop
extreme outliers. The means are computed globally on purpose — narrowing to a single
`REGION_NAME` would leave too few datapoints in each subregion.

Sparse-sample fallbacks:

- a region with fewer than `MIN_REGION_DATAPOINTS` unique pipelines uses the **global** mean
- a subregion with fewer than `MIN_SUBREGION_DATAPOINTS` unique pipelines uses its (possibly
  already-globalized) region's mean

Second, per-pipeline `CostUSDEstimate = LengthKnownKmByCountry × subregion_mean_cost_per_km`,
overwritten by the row's actual `LengthKnownKmByCountry × CostUSDPerKm` wherever both
populated. Aggregated into capex (USD billions) by status × country and status × region.

In [109]:
# Cost-per-km region/subregion means are built from the full global fuel set so that
# small REGION_NAME filters don't shrink the sample size — only the final capex pivot is
# restricted to the chosen region.
country_ratios_fuel_df = country_ratios_df.loc[country_ratios_df["Fuel"].isin(FUEL_OPTIONS)]

cost_df = pipes_df_orig.loc[pipes_df_orig["CostUSDPerKm"].notna()]
if cost_df.empty:
    raise RuntimeError(
        f"no {FUEL_LABEL} pipelines have a numeric CostUSDPerKm — cost estimates cannot run"
    )

q_lo, q_hi = cost_df["CostUSDPerKm"].quantile([0.025, 0.975])
print(f"CostUSDPerKm 2.5% / 97.5% quantiles: {q_lo:,.0f} / {q_hi:,.0f}")

# country-ratio rows that contribute to the regional means: need both a known
# CostUSDPerKm and a known length, with cost inside the trimmed window
ratios_with_cost = country_ratios_fuel_df.loc[
    country_ratios_fuel_df["CostUSDPerKm"].notna()
    & country_ratios_fuel_df["LengthKnownKmByCountry"].notna()
    & country_ratios_fuel_df["CostUSDPerKm"].between(q_lo, q_hi, inclusive="neither")
]
print(f"country-ratio rows feeding the cost averages: {len(ratios_with_cost):,}")

# global region/subregion lists & subregion→region lookup come from the full country
# dictionary — the regional means are global; the REGION_NAME filter is applied later
region_dict_clean = region_df_orig.loc[
    (region_df_orig["Region"] != "--") & (region_df_orig["SubRegion"] != "--")
]
region_list_global = sorted(region_dict_clean["Region"].dropna().unique())
subregion_list_global = sorted(region_dict_clean["SubRegion"].dropna().unique())
dict_subregion_region = dict(
    zip(region_dict_clean["SubRegion"], region_dict_clean["Region"])
)


def cost_table(level_col: str, level_values: list[str]) -> pd.DataFrame:
    """Mean CostUSDPerKm and unique-ProjectID count, grouped by `level_col`
    (Region or SubRegion)."""
    grouped = ratios_with_cost.groupby(level_col)
    out = pd.DataFrame(
        index=pd.Index(level_values, name=level_col),
        columns=["CostUSDPerKm", "DataPoints"],
        dtype=float,
    )
    out["CostUSDPerKm"] = grouped["CostUSDPerKm"].mean()
    out["DataPoints"] = grouped["ProjectID"].nunique()
    out["DataPoints"] = out["DataPoints"].fillna(0).astype(int)
    return out


pipes_costs_region_df = cost_table("Region", region_list_global)
pipes_costs_subregion_df = cost_table("SubRegion", subregion_list_global)

# Sparse-sample fallbacks. A region with too few datapoints is replaced by the global
# mean; a subregion with too few datapoints is replaced by its (possibly-now-global)
# region mean. The thresholds are inclusive — DataPoints < MIN_* triggers the fallback.
MIN_REGION_DATAPOINTS = 5
MIN_SUBREGION_DATAPOINTS = 5
global_mean_cost = ratios_with_cost["CostUSDPerKm"].mean()
print(f"global mean CostUSDPerKm (post-trim): {global_mean_cost:,.0f}")

sparse_region_mask = (
    pipes_costs_region_df["DataPoints"] < MIN_REGION_DATAPOINTS
) | pipes_costs_region_df["CostUSDPerKm"].isna()
if sparse_region_mask.any():
    print(
        f"replacing region mean with global for {sparse_region_mask.sum()} region(s) "
        f"with < {MIN_REGION_DATAPOINTS} datapoints: "
        f"{pipes_costs_region_df.index[sparse_region_mask].tolist()}"
    )
pipes_costs_region_df.loc[sparse_region_mask, "CostUSDPerKm"] = global_mean_cost

sparse_subregion_mask = (
    pipes_costs_subregion_df["DataPoints"] < MIN_SUBREGION_DATAPOINTS
) | pipes_costs_subregion_df["CostUSDPerKm"].isna()
if sparse_subregion_mask.any():
    print(
        f"replacing subregion mean with region mean for {sparse_subregion_mask.sum()} "
        f"subregion(s) with < {MIN_SUBREGION_DATAPOINTS} datapoints"
    )
for sr in pipes_costs_subregion_df.index[sparse_subregion_mask]:
    pipes_costs_subregion_df.loc[sr, "CostUSDPerKm"] = pipes_costs_region_df.loc[
        dict_subregion_region[sr], "CostUSDPerKm"
    ]

# write per-km tables to Excel in USD millions per km for readability
(pipes_costs_region_df.assign(CostUSDMillionsPerKm=lambda d: d["CostUSDPerKm"] / 1e6)
    .drop(columns="CostUSDPerKm")
    .sort_values("CostUSDMillionsPerKm", ascending=False)
    .to_excel(excel_writer, sheet_name="Cost per km by region"))
(pipes_costs_subregion_df.assign(CostUSDMillionsPerKm=lambda d: d["CostUSDPerKm"] / 1e6)
    .drop(columns="CostUSDPerKm")
    .sort_values("CostUSDMillionsPerKm", ascending=False)
    .to_excel(excel_writer, sheet_name="Cost per km by subregion"))

pipes_costs_subregion_df.sort_values("CostUSDPerKm", ascending=False)


CostUSDPerKm 2.5% / 97.5% quantiles: 76,358 / 16,944,129
country-ratio rows feeding the cost averages: 222
global mean CostUSDPerKm (post-trim): 2,955,804
replacing subregion mean with region mean for 7 subregion(s) with < 5 datapoints


,CostUSDPerKm,DataPoints
SubRegion,,
Southern Asia,5.497063e+06,8
Sub-Saharan Africa,4.106876e+06,7
Eastern Europe,3.504440e+06,13
Western Asia,3.205157e+06,19
South-eastern Asia,3.069484e+06,4
Northern America,3.067826e+06,36
Eastern Asia,3.026070e+06,68
Latin America and the Caribbean,2.805419e+06,14
Southern Europe,2.535515e+06,4


In [110]:
# Per-pipeline cost estimate, then capex pivot for the selected REGION_NAME / FUEL bucket.
# Default estimate: known-length × subregion-mean per-km cost.
# Overwrite: rows that already have BOTH a known length and a known per-km cost on the
# pipeline itself get the actual product.
country_ratios_with_capex = country_ratios_df_subset.copy()
country_ratios_with_capex["CostUSDEstimate"] = (
    country_ratios_with_capex["LengthKnownKmByCountry"]
    * country_ratios_with_capex["SubRegion"].map(pipes_costs_subregion_df["CostUSDPerKm"])
)
known_cost_mask = (
    country_ratios_with_capex["LengthKnownKmByCountry"].notna()
    & country_ratios_with_capex["CostUSDPerKm"].notna()
)
country_ratios_with_capex.loc[known_cost_mask, "CostUSDEstimate"] = (
    country_ratios_with_capex.loc[known_cost_mask, "LengthKnownKmByCountry"]
    * country_ratios_with_capex.loc[known_cost_mask, "CostUSDPerKm"]
)


def pivot_capex(group_cols, index) -> pd.DataFrame:
    """Sum CostUSDEstimate by Status × group_cols, reshape to wide form, in USD billions."""
    grouped = (
        country_ratios_with_capex.groupby([*group_cols, "Status"])["CostUSDEstimate"]
        .sum()
        .unstack("Status")
        .reindex(columns=STATUS_LIST)
        .reindex(index=index)
        .fillna(0)
    ) / 1e9
    grouped[IN_DEV_COL] = grouped[["proposed", "construction"]].sum(axis=1)
    return grouped[EXCEL_STATUS_LIST]


capex_by_country = pivot_capex(["Country"], country_list)
capex_by_country.index.name = "Country"

capex_by_region = pivot_capex(["Region", "SubRegion"], multiindex_region_subregion)
capex_by_region.index.names = ["Region", "Subregion"]

capex_by_country.loc["Total"] = capex_by_country.sum(axis=0).values
# full 2-level key — a flat "Total" label would collapse the MultiIndex to tuples
capex_by_region.loc[("Total", ""), :] = capex_by_region.sum(axis=0).values

# drop countries with no capex in any status, then blank out zeros for display
capex_by_country = capex_by_country.loc[~(capex_by_country == 0).all(axis=1)]
capex_by_country = capex_by_country.replace(0, "")
capex_by_region = capex_by_region.replace(0, "")

capex_by_region.to_excel(excel_writer, sheet_name="Capex USD billions by region")
capex_by_country.to_excel(excel_writer, sheet_name="Capex USD billions by country")
capex_by_region


Status                                     proposed construction  \
Region   Subregion                                                 
Africa   Northern Africa                   0.627414     3.180027   
         Sub-Saharan Africa               12.186884          5.0   
Americas Latin America and the Caribbean   3.380018                
         Northern America                  5.305551     2.405983   
Asia     Central Asia                      2.919533                
         Eastern Asia                      7.524009     2.926874   
         South-eastern Asia                                        
         Southern Asia                     0.566197    18.187986   
         Western Asia                     14.256755     5.421196   
Europe   Eastern Europe                    7.766629      6.16459   
         Northern Europe                                           
         Southern Europe                   0.407116                
         Western Europe                                            
Oceania  Australia and New Zealand                                 
         Melanesia                                                 
         Micronesia                                                
         Polynesia                                                 
Total                                     54.940106    43.286655   

Status                                   in development (proposed + construction)  \
Region   Subregion                                                                  
Africa   Northern Africa                                                 3.807441   
         Sub-Saharan Africa                                             17.186884   
Americas Latin America and the Caribbean                                 3.380018   
         Northern America                                                7.711533   
Asia     Central Asia                                                    2.919533   
         Eastern Asia                                                   10.450883   
         South-eastern Asia                                                         
         Southern Asia                                                  18.754183   
         Western Asia                                                    19.67795   
Europe   Eastern Europe                                                 13.931219   
         Northern Europe                                                            
         Southern Europe                                                 0.407116   
         Western Europe                                                             
Oceania  Australia and New Zealand                                                  
         Melanesia                                                                  
         Micronesia                                                                 
         Polynesia                                                                  
Total                                                                   98.226761   

Status                                      shelved   cancelled    operating  \
Region   Subregion                                                             
Africa   Northern Africa                   0.144707    1.309752    40.529887   
         Sub-Saharan Africa                            6.160314    54.982291   
Americas Latin America and the Caribbean       1.65    9.331483    73.297109   
         Northern America                             120.55714    366.78161   
Asia     Central Asia                                  1.043652    11.259436   
         Eastern Asia                      2.899455    6.179274    91.494807   
         South-eastern Asia                                0.38     5.651317   
         Southern Asia                      1.82556   24.847729    98.527992   
         Western Asia                      7.692377    2.884641    78.052915   
Europe   Eastern Europe                    1.530985   10.1

## save Excel

In [111]:
excel_writer.close()
print(f"wrote {output_path}")

wrote /Users/baird/Dropbox/_git_ALL/_github-repos-gem/goit-ggit-data-ops/scripts/data-release-summary-sheets/2026-q2-oil-pipelines/GOIT-Summary-Sheets-Oil-2026-06-15.xlsx


## landing-page stats

In [112]:
# pipes_df_touse is already subset to the chosen fuel bucket and region
print(f"{len(pipes_df_touse):>6,d} {FUEL_LABEL} pipeline projects tracked")
print(f"{pipes_df_touse['LengthMergedKm'].sum() / 1e6:>6.3f} M km tracked")

 1,634 Oil pipeline projects tracked
 0.487 M km tracked


In [113]:
in_dev = pipes_df_touse.loc[pipes_df_touse["Status"].isin(["proposed", "construction"])]
print(f"{len(in_dev):>6,d} {FUEL_LABEL} pipeline projects in development (proposed + construction)")
print(f"{in_dev['LengthMergedKm'].sum() / 1e3:>6.1f} K km in development")

   152 Oil pipeline projects in development (proposed + construction)
  32.4 K km in development
